# Resume–Job Fit
## Classification: Traditional vs. Pretrained Representations
### ARTI 404 – Natural Language Processing

In [1]:
%pip install pandas numpy scikit-learn datasets sentence-transformers matplotlib

   ---------------------------------------- 0.0/608.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.3 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.3 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.3 kB ? eta -:--:--
   ---------------------------------------- 608.3/608.3 kB 1.0 MB/s  0:00:00
   ---------------------------------------- 0.0/780.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/780.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/780.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/780.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/780.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/780.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/780.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/780.3 kB ? eta -:--:--
   ------------------------- ------------ 524.3/780.3 k

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.
streamlit 1.51.0 requires pyarrow<22,>=7.0, but you have pyarrow 26.0.1 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 1. Setup

In [2]:
import random
import numpy as np
import pandas as pd
from datasets import load_dataset

# Fix the random seed so results are reproducible on every run
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_colwidth", 120)

#### 2. Data

In [3]:
# Load the dataset from Hugging Face
raw = load_dataset("cnamuangtoun/resume-job-description-fit")

# Merge the original train and test parts into one table,
# because we will create our own resume-based split later
df = pd.concat(
    [raw["train"].to_pandas(), raw["test"].to_pandas()],
    ignore_index=True
)

print("Shape:", df.shape)
df.head()

c:\Users\rvl19\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rvl19\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

c:\Users\rvl19\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rvl19\.cache\huggingface\hub\datasets--cnamuangtoun--resume-job-description-fit. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

Shape: (8000, 3)


,resume_text,job_description_text,label
0,SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales profess...,Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for o...,No Fit
1,Professional SummaryCurrently working with Caterpillar as an contract employee Active in several NPI and CPI project...,At Salas OBrien we tell our clients that were engineered for impact. This passion for making a difference applies ju...,No Fit
2,"SummaryI started my construction career in June of 2017 in Jacksonville, Florida as an Electrical Apprentice working...",Schweitzer Engineering Laboratories (SEL) Infrastructure Defense Division seeks a talented individual for a cybersec...,No Fit
3,SummaryCertified Electrical Foremanwith thirteen years of experience in the implementation of electrical systems on ...,"Mizick Miller & Company, Inc. is looking for a dynamic individual to join our team!This is a great opportunity for t...",No Fit
4,SummaryWith extensive experience in business/requirement analysis with SDLC processing. Experience includes project ...,Life at Capgemini\nCapgemini supports all aspects of your well-being throughout the changing stages of your life and...,No Fit


In [4]:
# Class counts
print("Label counts:")
print(df["label"].value_counts(), "\n")

# Unique resumes and job descriptions
print("Unique resumes:", df["resume_text"].nunique())
print("Unique job descriptions:", df["job_description_text"].nunique(), "\n")

# Empty texts and fully duplicated rows
print("Empty resumes:", (df["resume_text"].str.strip() == "").sum())
print("Empty job descriptions:", (df["job_description_text"].str.strip() == "").sum())
print("Duplicated rows:", df.duplicated().sum(), "\n")

# Text lengths in words
df["resume_len"] = df["resume_text"].str.split().str.len()
df["job_len"] = df["job_description_text"].str.split().str.len()
print(df[["resume_len", "job_len"]].describe().round(0))

Label counts:
label
No Fit           4000
Potential Fit    2000
Good Fit         2000
Name: count, dtype: int64 

Unique resumes: 643
Unique job descriptions: 351 

Empty resumes: 0
Empty job descriptions: 0
Duplicated rows: 1 

       resume_len  job_len
count      8000.0   8000.0
mean        704.0    378.0
std         387.0    250.0
min          84.0     11.0
25%         488.0    169.0
50%         618.0    326.0
75%         809.0    552.0
max        3134.0   1254.0


#### Resume-based split
All pairs that share the same resume are kept in the same split (train / dev / test), so the model is always tested on unseen resumes.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

# Remove the single fully duplicated row
df = df.drop_duplicates().reset_index(drop=True)

# Give every unique resume an id, used as the group for splitting
df["resume_id"] = df["resume_text"].factorize()[0]

# First split: 70% train, 30% temporary (by resume)
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss.split(df, groups=df["resume_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

# Second split: the 30% is divided equally into dev and test (by resume)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
dev_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["resume_id"]))
dev_df = temp_df.iloc[dev_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

# Sizes and label distribution of each split
for name, part in [("Train", train_df), ("Dev", dev_df), ("Test", test_df)]:
    print(f"{name}: {len(part)} pairs, {part['resume_id'].nunique()} resumes")
    print(part["label"].value_counts(normalize=True).round(2).to_dict(), "\n")

Train: 5683 pairs, 450 resumes
{'No Fit': 0.5, 'Good Fit': 0.25, 'Potential Fit': 0.25} 

Dev: 1208 pairs, 96 resumes
{'No Fit': 0.51, 'Good Fit': 0.25, 'Potential Fit': 0.23} 

Test: 1108 pairs, 97 resumes
{'No Fit': 0.49, 'Potential Fit': 0.27, 'Good Fit': 0.24} 



In [7]:
# Check that no resume appears in more than one split
train_r = set(train_df["resume_id"])
dev_r = set(dev_df["resume_id"])
test_r = set(test_df["resume_id"])
assert not (train_r & dev_r) and not (train_r & test_r) and not (dev_r & test_r)
print("No resume is shared between splits.")

# Job descriptions may repeat across splits (reported as a limitation)
train_j = set(train_df["job_description_text"])
test_jobs = test_df["job_description_text"]
print("Test pairs whose job description also appears in train:",
    test_jobs.isin(train_j).sum(), "of", len(test_df))

No resume is shared between splits.
Test pairs whose job description also appears in train: 1104 of 1108


#### Lexical overlap and the low-overlap subset
Overlap is measured with Jaccard similarity between the sets of normalized content words (lowercased, stop words removed) of each CV and its job description. The low-overlap threshold is chosen on the development set only.

In [9]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def normalize_text(text):
    # Split words that were glued together, e.g. "SummaryHighly" -> "Summary Highly"
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    # Lowercase and keep letters, numbers and spaces only
    text = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    return re.sub(r"\s+", " ", text).strip()

def content_words(text):
    # Set of meaningful words: no stop words, no single characters
    return {w for w in normalize_text(text).split()
            if w not in ENGLISH_STOP_WORDS and len(w) > 1}

def jaccard(text_a, text_b):
    a, b = content_words(text_a), content_words(text_b)
    return len(a & b) / len(a | b) if (a | b) else 0.0

# Compute the overlap for every pair in each split
for part in [train_df, dev_df, test_df]:
    part["jaccard"] = [jaccard(r, j) for r, j in
                    zip(part["resume_text"], part["job_description_text"])]

# Average overlap per label in the training set
print(train_df.groupby("label")["jaccard"].mean().round(3))

label
Good Fit         0.076
No Fit           0.061
Potential Fit    0.066
Name: jaccard, dtype: float64


In [10]:
# Threshold = the 25th percentile of overlap in the development set
LOW_OVERLAP_THRESHOLD = dev_df["jaccard"].quantile(0.25)
print("Low-overlap threshold:", round(LOW_OVERLAP_THRESHOLD, 3))

# Apply the same threshold to the test set
test_df["low_overlap"] = test_df["jaccard"] <= LOW_OVERLAP_THRESHOLD
low_test_df = test_df[test_df["low_overlap"]].reset_index(drop=True)

print("Low-overlap test pairs:", len(low_test_df), "of", len(test_df))
print(low_test_df["label"].value_counts())

Low-overlap threshold: 0.047
Low-overlap test pairs: 294 of 1108
label
No Fit           174
Potential Fit     81
Good Fit          39
Name: count, dtype: int64
